# Import Libraries

In [ ]:
import os
import time
import pickle
import random
import itertools

# Analisi dati e Visualizzazione
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn: Preprocessing e Dimensionality Reduction
from sklearn.preprocessing import (
    MinMaxScaler,
    StandardScaler,
    LabelEncoder,
    OneHotEncoder
)
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.utils.class_weight import compute_class_weight

# Scikit-learn: Model Selection e Pipeline
from sklearn.model_selection import (
    train_test_split,
    PredefinedSplit,
    GridSearchCV,
    RandomizedSearchCV,
    ParameterGrid
)
from sklearn.pipeline import Pipeline

# Scikit-learn: Modelli di Machine Learning Classico
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.tree import plot_tree

# Imbalanced Learning (SMOTE e Pipeline compatibile)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# Metriche di Valutazione
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    roc_auc_score,
    roc_curve,
    PrecisionRecallDisplay
)

# Deep Learning (PyTorch) e Monitoraggio
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter

import datetime

# Our packages
from utils import *
from plot import *

from sklearn.base import BaseEstimator, TransformerMixin

# Global Variables

In [ ]:
SEED = 42
FILENAME = "../../data/train.csv"

# Cerca la GPU
if torch.backends.mps.is_available():
    print("MPS device is available.")
    device = torch.device("mps")
elif torch.cuda.is_available():
    print("CUDA device is available.")
    device = torch.device("cuda")
else:
    print("No GPU acceleration available.")
    device = torch.device("cpu")

In [ ]:
def fix_random(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

fix_random(SEED)

# Load the dataset

In [ ]:
df = pd.read_csv(FILENAME, encoding='ISO-8859-1', sep=",")

rows = df.shape[0]
cols = df.shape[1]
print("# Righe: " + str(rows)+ " # Colonne: "+str(cols) + "\n")

# Preprocessing

## 1. Remove duplicates rows and columns

In [ ]:
# Individua se esistono colonne con lo stesso nome
# Se esistono, allora se le colonne sono duplicati perfetti, droppiamo il duplicato
# Se esistono, ma nono sono perfetti duplicati, per intervenire consciamente sarebbe necessario avere maggior domain knowledge
feature_list = df.columns.to_list()
has_duplicate_cols = len(feature_list) != len(set(feature_list))
print("Ci sono colonne con lo stesso nome?", has_duplicate_cols)

if has_duplicate_cols:
    df2 = df.T.drop_duplicates().T


# Rimuovi righe duplicate
df.drop_duplicates(inplace=True)


##################################################
print("Nuovo # Righe: " + str(rows)+ " Nuovo # Colonne: "+str(cols) + "\n")


## 2. Label extraction and train-test splitting

In [ ]:
X = df.drop(columns=["grade"])
y = df["grade"]

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)

## 3. Pipeline

In [ ]:
class ColumnDropper(BaseEstimator, TransformerMixin):
    """ Drop generic columns """
    def __init__(self):
        self.cols_to_drop = []

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        return X.drop(columns=[col for col in self.cols_to_drop if col in X.columns])
    

class HighNanDropper(BaseEstimator, TransformerMixin):
    """ Rimuove colonne con alto numero di NaN"""
    
    def __init__(self, threshold=90):
        self.threshold = threshold
        self.cols_to_drop = []

    def fit(self, X, y=None):
        nan_ratio = X.isna().mean() * 100
        self.cols_to_drop = nan_ratio[nan_ratio > self.threshold].index.tolist()
        return self

    def transform(self, X):
        X = X.copy()
        return X.drop(columns=[col for col in self.cols_to_drop if col in X.columns])
    

class HighlyCorrelatedDropper(BaseEstimator, TransformerMixin):
    """" Rimozione delle feature ridondanti identificate (High Correlation > threshold) """
    def __init__(self, threshold=0.95):
        self.threshold = threshold
        self.correlated_cols = []

    def fit(self, X, y=None):
        numeric_df = X.select_dtypes(include=[np.number])
        corr_matrix = numeric_df.corr().abs()
        # Selezioniamo il triangolo superiore della matrice; k=1 esclude la diagonale principale
        upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        self.correlated_cols = [column for column in upper_tri.columns if any(upper_tri[column] > self.threshold)]
        return self

    def transform(self, X):
        X = X.copy()
        return X.drop(columns=[col for col in self.correlated_cols if col in X.columns])
    

class NumericExtractor(BaseEstimator, TransformerMixin):
    """ Estrae feature numeriche da stringhe (36/60 mesi, polishing anni di carriera...) """
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        for col in X.columns:
            X[col] = X[col].replace({ '< 1': 0}).astype(float)
            X[col] = X[col].astype(str).str.extract(r"(\d+)").astype(float)
        return X
    
    
class FeatureAverager(BaseEstimator, TransformerMixin):
    """Media di n colonne e rimuove gli originali"""
    def __init__(self, new_name='new_avg_col'):
        self.new_name = new_name

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        # Calcoliamo la media lungo l'asse delle righe (axis=1)
        X[self.new_name] = X.mean(axis=1)
        return X.drop(columns=[col for col in self.columns_to_drop if col in X.columns])
    

class MonthDifferenceCalculator(BaseEstimator, TransformerMixin):
    """Media due colonne e rimuove gli originali"""
    def __init__(self, new_name='new_avg_col'):
        self.new_name = new_name

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        # Calcoliamo la media lungo l'asse delle righe (axis=1)
        X[self.new_name] = X.mean(axis=1)
        return X.drop(columns=[col for col in self.columns_to_drop if col in X.columns])
    

    class DateDifferenceTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, reference_col, target_cols=None, date_format='%b-%Y', drop_original=True):
        """
        Calculates the difference in months between a Reference Column and Target Columns.
        Formula: Reference - Target
        
        Args:
            reference_col (str): The column to subtract FROM (e.g., 'loan_issue_date').
            target_cols (list or str): Column(s) to subtract (e.g., 'earliest_cr_line').
            date_format (str): The format to parse dates (e.g., '%b-%Y').
            drop_original (bool): If True, drops the used date columns after calculation.
        """
        self.reference_col = reference_col
        # Ensure target_cols is a list
        self.target_cols = [target_cols] if isinstance(target_cols, str) else target_cols
        self.date_format = date_format
        self.drop_original = drop_original

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        if self.reference_col in X.columns:
            ref_series = pd.to_datetime(X[self.reference_col], format=self.date_format, errors='coerce')
        else:
            raise ValueError(f"Reference column '{self.reference_col}' not found in dataset.")

        # 2. Loop through Target Columns
        cols_to_drop = []
        
        if self.target_cols:
            targets = [c for c in self.target_cols if c in X.columns]
        else:
            # OPTIONAL: Auto-detect all other object columns if target_cols is None (Risky but possible)
            targets = [] 

        for col in targets:
            # Convert target to datetime
            target_series = pd.to_datetime(X[col], format=self.date_format, errors='coerce')
            
            # 3. Calculate Months Difference
            # Formula: (YearRef - YearTarget) * 12 + (MonthRef - MonthTarget)
            new_col_name = f"months_since_{col}"
            
            X[new_col_name] = (
                (ref_series.dt.year - target_series.dt.year) * 12 +
                (ref_series.dt.month - target_series.dt.month)
            )
            
            # Fill NaNs created by parse errors with a placeholder (optional) or leave as NaN
            # X[new_col_name] = X[new_col_name].fillna(-1) 

            cols_to_drop.append(col)

        # 4. Cleanup
        if self.drop_original:
            # Drop targets + reference column
            all_drops = cols_to_drop + [self.reference_col]
            # Only drop columns that exist
            X = X.drop(columns=[c for c in all_drops if c in X.columns], errors='ignore')
            
        return X

In [ ]:
# Configuration FOR RANDOM FOREST: we use state and drop zip
redundant_cols = ['loan_title', "borrower_address_zip", "borrower_address_state"] 
binary_cols = ["loan_payment_plan_flag", "listing_initial_status", "application_type_label",
               "hardship_flag_indicator", "disbursement_method_type", "debt_settlement_flag_indicator"]
one_hot_encoding_cols = ["borrower_housing_ownership_status", "borrower_income_verification_status",
                       "loan_status_current_code", "loan_purpose_category"]
extract_fields = ["loan_contract_term_months", "borrower_profile_employment_length"]
date_fields = ["loan_issue_date", "credit_history_earliest_line", "last_payment_date", "last_credit_pull_date"]



In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('extract', NumericExtractor(), extract_fields),
        ('categorical', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), one_hot_encoding_cols),
        ('date', CyclicalDateEncoder(), date_fields),
        ('binary', BinaryModeEncoder(), binary_cols),

        ('drop_redundant', 'drop', redundant_cols)
    ], 
    remainder="passthrough"
)

full_pipeline = Pipeline([
    ('dropper', HighMissingDropper(threshold=20)),
    ('prep', preprocessor),
    ('rf', RandomForestClassifier(class_weight='balanced', n_jobs=-1))
])

param_grid_rf = {
    'rf__max_depth': [20, None],
    'rf__n_estimators': [100, 200],
    'rf__min_samples_leaf': [1, 5]
}

""" 
param_grid_rf = {
    'clf__n_estimators': [250, 300],
    'clf__max_features': ['sqrt'],      # 'log2'
    'clf__criterion': ['gini', 'log_loss'],
    'clf__max_depth': [10, 20]
}
 """

scoring = {
    'acc': 'accuracy',
    'balanced_acc': 'balanced_accuracy',
    'f1_weighted': 'f1_weighted'
}

grid_search = GridSearchCV(
    estimator=full_pipeline,
    param_grid=param_grid_rf,
    cv=3,
    scoring=scoring,
    refit='balanced_acc', 
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X, y)

In [ ]:
# Best parameters found
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV score: {grid_search.best_score_:.4f}")

print("--"*20)
results = grid_search.cv_results_
best_index = grid_search.best_index_

# Now these keys will exist because you used the dictionary scoring
print(f"Best Model Results (Refit on F1_Weighted):")
print(f"Mean Accuracy: {results['mean_test_acc'][best_index]:.4f}")
print(f"Mean Balanced Acc: {results['mean_test_balanced_acc'][best_index]:.4f}")
print(f"Mean F1 Weighted: {results['mean_test_f1_weighted'][best_index]:.4f}")

print("--"*20)